# PSEO Data Extraction -- CIP 60 (Unusable)

Extracts rows from `pseoe_all.csv` where:
- `agg_level_pseo == 46` → Degree Level × CIP 2-digit × Graduation Cohort Start Year × Institution ID
- `cipcode` is one of `11`, `13`, or `60`

**CIP codes:**
- `11` — Computer and Information Sciences and Support Services
- `13` — Education
- `60` — Health Professions Residency/Fellowship Programs

Reads in chunks to avoid loading the full ~64 MB file into memory at once.

In [2]:
import pandas as pd
import os

INPUT_FILE  = "pseoe_all.csv"
OUTPUT_FILE = "pseo_agg46_cip11_13_60.csv"
CHUNK_SIZE  = 50_000

TARGET_AGG_LEVEL = 46
TARGET_CIP_CODES = {"11", "13", "60"}   # 2-digit CIP codes as strings

print(f"Reading '{INPUT_FILE}' in chunks of {CHUNK_SIZE:,} rows...")

Reading 'pseoe_all.csv' in chunks of 50,000 rows...


In [4]:
dtype_settings = {
    'agg_level_pseo': int,
    'institution': str,    # Source State FIPS
    'cipcode': str,
    'degree_level': str,
    'grad_cohort': str,    # Keep as string to handle '2001'
    'geography': str,      # Destination Division
    'industry': str,       # NAICS Sector
    'y1_grads_emp': float, 'y1_grads_emp_instate': float,
    'y5_grads_emp': float, 'y5_grads_emp_instate': float,
    'y10_grads_emp': float, 'y10_grads_emp_instate': float,
    'status_y1_grads_emp': float,
    'status_y5_grads_emp': float,
    'status_y10_grads_emp': float,
    'y1_grads_nme': float, 'y5_grads_nme': float, 'y10_grads_nme': float
}

chunks = []

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings)):
    # Normalise cipcode: strip whitespace and zero-pad to 2 digits
    chunk["cipcode"] = chunk["cipcode"].str.strip().str.zfill(2)

    filtered = chunk[
        (chunk["agg_level_pseo"] == TARGET_AGG_LEVEL) &
        (chunk["cipcode"].isin(TARGET_CIP_CODES))
    ]

    if not filtered.empty:
        chunks.append(filtered)

    if (i + 1) % 5 == 0:
        rows_scanned = (i + 1) * CHUNK_SIZE
        print(f"  Scanned ~{rows_scanned:,} rows so far...")

print("Done scanning.")

/var/folders/tm/62knn7lj32q35mqf_5r316700000gn/T/ipykernel_91689/2821426274.py:20: DtypeWarning: Columns (0: cip_level) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings)):
/var/folders/tm/62knn7lj32q35mqf_5r316700000gn/T/ipykernel_91689/2821426274.py:20: DtypeWarning: Columns (0: cip_level) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings)):
/var/folders/tm/62knn7lj32q35mqf_5r316700000gn/T/ipykernel_91689/2821426274.py:20: DtypeWarning: Columns (0: cip_level) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings)):


  Scanned ~250,000 rows so far...
  Scanned ~500,000 rows so far...
Done scanning.


In [5]:
if chunks:
    df = pd.concat(chunks, ignore_index=True)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved {len(df):,} rows to '{OUTPUT_FILE}'")
else:
    print("No matching rows found. Double-check the filter criteria.")

Saved 12,891 rows to 'pseo_agg46_cip11_13_60.csv'


In [6]:
# Quick sanity check on the extracted file
df = pd.read_csv(OUTPUT_FILE, dtype={"cipcode": str})

print("Shape:", df.shape)
print()
print("Unique agg_level_pseo values:", df["agg_level_pseo"].unique())
print()
print("Row counts by cipcode:")
print(df["cipcode"].value_counts().sort_index())
print()
print("Row counts by degree_level:")
print(df["degree_level"].value_counts().sort_index())
print()
print(df.head(3).to_string())

Shape: (12891, 36)

Unique agg_level_pseo values: [46]

Row counts by cipcode:
cipcode
11    6994
13    5889
60       8
Name: count, dtype: int64

Row counts by degree_level:
degree_level
1     1390
2     1258
3     2660
4       55
5     4686
6        9
7     1985
17     805
18      43
Name: count, dtype: int64

   agg_level_pseo inst_level institution  degree_level  cip_level cipcode  grad_cohort  grad_cohort_years geo_level  geography ind_level  industry  y1_p25_earnings  y1_p50_earnings  y1_p75_earnings  y1_grads_earn  y5_p25_earnings  y5_p50_earnings  y5_p75_earnings  y5_grads_earn  y10_p25_earnings  y10_p50_earnings  y10_p75_earnings  y10_grads_earn  y1_ipeds_count  y5_ipeds_count  y10_ipeds_count  status_y1_earnings  status_y1_grads_earn  status_y5_earnings  status_y5_grads_earn  status_y10_earnings  status_y10_grads_earn  status_y1_ipeds_count  status_y5_ipeds_count  status_y10_ipeds_count
0              46          I    00105100             5          2      11         2001    

In [7]:
print("Row counts by Geography:")
print(df["geography"].value_counts().sort_index())

Row counts by Geography:
geography
0    12891
Name: count, dtype: int64
